# Ordering the data

## Import

importing libraries as well as connecting to the google drive, for the dataset

In [2]:
import pandas as pd
import numpy as np

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#Dataframe view

df = pd.read_csv('/content/drive/MyDrive/data /SQL-Final project/data/hotel_bookings.csv')
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [5]:
#info about columns:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

## Fixing

We have couple of problems in the original dataset, which should be solved

Here are they, listed along with the problem, and the solution:

| Column                    | Problem               | Fix                    |
| ------------------------- | --------------------- | ---------------------- |
| `children`                | float64, 4 nulls      | fillna(0), cast to int |
| `country`                 | 488 nulls             | fillna('Unknown')      |
| `arrival_date_month`      | string ("July")       | map to number (7)      |
| `reservation_status_date` | string, not date      | convert to datetime    |
| `agent`                   | float64, 16340 nulls  | fillna(0), cast to int |
| `company`                 | float64, 112593 nulls | fillna(0), cast to int |


In [6]:
#fixing children column:
#problems: 4 nulls, float64

df['children'] = df['children'].fillna(0)

df['children'] = df['children'].astype(int)

# Verify the change
print(df['children'].dtype)
print(df['children'].isnull().sum())

int64
0


In [7]:
#fixing country column:
#problems: 488 nulls

df['country'] = df['country'].fillna('Unknown')

#Verifying the change
print(df['country'].isnull().sum())

0


In [8]:
#fixing arrival_date_month
#problem: mapping each month to integer type

month_map = {'January': 1, 'February': 2, 'March': 3, 'April': 4,'May': 5,'June': 6,
             'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12}


df['arrival_date_month'] = df['arrival_date_month'].map(month_map)

df['arrival_date_month'] = df['arrival_date_month'].astype(int)

# verifying the change
print(df['arrival_date_month'].unique())

[ 7  8  9 10 11 12  1  2  3  4  5  6]


In [9]:
# fixing: reservation_status_date
# problem: not datetime object, string
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])

# Verify the change
print(df['reservation_status_date'].dtype)


df['reservation_status_date'].head()

datetime64[ns]


,reservation_status_date
0,2015-07-01
1,2015-07-01
2,2015-07-02
3,2015-07-02
4,2015-07-03


In [10]:
#fixing agent column
#problem: float64, 16340 nulls


df['agent'] = df['agent'].fillna(0)

df['agent'] = df['agent'].astype(int)

print(f"Nulls remaining: {df['agent'].isnull().sum()}")
print(f"New data type: {df['agent'].dtype}")

Nulls remaining: 0
New data type: int64


In [11]:
#fixing company column:
#problem: float64, 112593 nulls

df['company'] = df['company'].fillna(0)

df['company'] = df['company'].astype(int)


print(f"Total nulls in company: {df['company'].isnull().sum()}")
print(f"Column data type: {df['company'].dtype}")

Total nulls in company: 0
Column data type: int64


### Recheck

We decided to recheck each column, and see if there any other possible problems


|Column|Concern|
|---|---|
|`adr`|float — check for negative values (impossible price)|
|`adults`|check for 0 adults with no children/babies (ghost booking)|
|`stays_in_weekend_nights` + `stays_in_week_nights`|check if both are 0 (zero-night stay makes no sense)|

#### adr

In [12]:
# adr

print(f"Minimum ADR: {df['adr'].min()}")

Minimum ADR: -6.38


In [13]:
# there is at least one row, with negative value, so we should check for the amount of them
negative_adr = df[df['adr'] < 0]
print(f"Number of negative ADR records: {len(negative_adr)}")
negative_adr

Number of negative ADR records: 1


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
14969,Resort Hotel,0,195,2017,3,10,5,4,6,2,...,No Deposit,273,0,0,Transient-Party,-6.38,0,0,Check-Out,2017-03-15


In [14]:
# since there is only one row, it will be better to just delete the column
df = df[df['adr'] >= 0]

In [15]:
# we also need to see if there any outliers:
print(f"Maximum ADR: {df['adr'].max()}")

df['adr'].sort_values(ascending=False).head(10)

Maximum ADR: 5400.0


,adr
48515,5400.00
111403,510.00
15083,508.00
103912,451.50
13142,450.00
13391,437.00
39155,426.25
39568,402.00
39118,397.38
13323,392.00


In [16]:
# looks like the row 48515 has outlier adr
# deleting the row
df = df[df['adr'] < 5000]
df['adr'].sort_values(ascending=False).head(10)

,adr
111403,510.00
15083,508.00
103912,451.50
13142,450.00
13391,437.00
39155,426.25
39568,402.00
39118,397.38
39517,392.00
13323,392.00


#### adults

In [17]:
# we need to check if there are ghost bookings

ghost_bookings = df[(df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0)]

print(f"Number of ghost bookings found: {len(ghost_bookings)}")
ghost_bookings[['adults', 'children', 'babies']].head()

Number of ghost bookings found: 180


,adults,children,babies
2224,0,0,0
2409,0,0,0
3181,0,0,0
3684,0,0,0
3708,0,0,0


In [18]:
# removing 180 rows with ghost bookings
df = df[~((df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0))]

ghost_bookings = df[(df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0)]
print(f"Number of ghost bookings found: {len(ghost_bookings)}")

Number of ghost bookings found: 0


#### stays_in_weekend_nights + stays_in_week_nights

In [19]:
# Filter for rows where both weekend and week nights are 0
zero_nights = df[(df['stays_in_weekend_nights'] == 0) & (df['stays_in_week_nights'] == 0)]

print(f"Number of zero-night stays found: {len(zero_nights)}")
zero_nights[['stays_in_weekend_nights', 'stays_in_week_nights', 'adr']].head()

Number of zero-night stays found: 645


,stays_in_weekend_nights,stays_in_week_nights,adr
0,0,0,0.0
1,0,0,0.0
167,0,0,0.0
168,0,0,0.0
196,0,0,0.0


In [20]:
#same as before, the number for them is really small, so it's better to just remove them
df = df[~((df['stays_in_weekend_nights'] == 0) & (df['stays_in_week_nights'] == 0))]

#check
zero_nights = df[(df['stays_in_weekend_nights'] == 0) & (df['stays_in_week_nights'] == 0)]
print(f"Number of zero-night stays found: {len(zero_nights)}")

Number of zero-night stays found: 0


In [21]:
print(f"Rows remaining: {len(df)}")

Rows remaining: 118563
